In [11]:
import pandas as pd
import numpy as np
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

### Preparação


In [6]:
from pathlib import Path
import hashlib
import json
from processamento import (
    CLASSES, ROTULOS_PASTAS, limpar_texto, tokenizar,
    transformar_textos, texto_para_tfidf, extrair_caracteristicas,
)

RAIZ = Path.cwd()
DATASET = RAIZ / "dataset_caed"
if not DATASET.is_dir():
    raise FileNotFoundError("Execute o notebook na raiz que contém dataset_caed.")
RESULTADOS = RAIZ / "resultados"
RESULTADOS.mkdir(exist_ok=True)
SEMENTE = 42

### Leitura e Auditoria

In [7]:
registros = []
for arquivo in sorted(DATASET.rglob("*.txt")):
    classe = ROTULOS_PASTAS[arquivo.parent.name]
    texto = limpar_texto(arquivo.read_text(encoding="utf-8-sig"))
    registros.append({
        "documento": arquivo.relative_to(DATASET).as_posix(),
        "texto": texto,
        "classe": classe,
        "n_palavras": len(tokenizar(texto)),
        "hash_texto": hashlib.sha256(texto.lower().encode("utf-8")).hexdigest(),
    })

df = pd.DataFrame(registros)
if df.empty:
    raise ValueError("Nenhum arquivo .txt encontrado.")
print("Total original:", len(df))
print(df.groupby("classe")["n_palavras"].agg(["count", "min", "median", "max"]))

vazios = df[df["n_palavras"] == 0].copy()
df = df[df["n_palavras"] > 0].copy()
rotulos_por_hash = df.groupby("hash_texto")["classe"].nunique()
hashes_conflitantes = rotulos_por_hash[rotulos_por_hash > 1].index
conflitos = df[df["hash_texto"].isin(hashes_conflitantes)].copy()
df = df[~df["hash_texto"].isin(hashes_conflitantes)].copy()
duplicatas = df[df.duplicated("hash_texto", keep="first")].copy()
df = df.drop_duplicates("hash_texto").reset_index(drop=True)

for nome, tabela in [("vazios", vazios), ("conflitos", conflitos),
                     ("duplicatas", duplicatas)]:
    tabela.drop(columns="texto").to_csv(RESULTADOS / f"{nome}.csv", index=False)
    print(nome, len(tabela), "arquivos")
print("Documentos restantes:", len(df))
assert set(df["classe"]) == set(CLASSES)

Total original: 2076
                       count  min  median  max
classe                                        
Ensino Fundamental I     297  295   326.0  579
Ensino Fundamental II    325  284   369.0  617
Ensino Médio             628  263   347.0  748
Ensino Superior          826  266   357.0  803
vazios 0 arquivos
conflitos 24 arquivos
duplicatas 4 arquivos
Documentos restantes: 2048


### Separação em treino, validação e teste

In [8]:
from sklearn.model_selection import train_test_split

treino, restante = train_test_split(
    df, test_size=0.30, stratify=df["classe"], random_state=SEMENTE,
)
validacao, teste = train_test_split(
    restante, test_size=0.50, stratify=restante["classe"], random_state=SEMENTE,
)
partes = {"treino": treino, "validacao": validacao, "teste": teste}
for a, b in [(treino, validacao), (treino, teste), (validacao, teste)]:
    assert set(a["documento"]).isdisjoint(b["documento"])
    assert set(a["hash_texto"]).isdisjoint(b["hash_texto"])

manifesto = pd.concat([
    tabela[["documento", "classe", "hash_texto"]].assign(conjunto=nome)
    for nome, tabela in partes.items()
])
manifesto.to_csv(RESULTADOS / "divisao_dados.csv", index=False)
print(pd.crosstab(manifesto["classe"], manifesto["conjunto"]))

conjunto               teste  treino  validacao
classe                                         
Ensino Fundamental I      44     206         44
Ensino Fundamental II     48     227         49
Ensino Médio              93     431         92
Ensino Superior          123     569        122


### Pipelines e comparação na validação

In [ ]:
def criar_representacao():
    return FeatureUnion([
        ("vocabulario", TfidfVectorizer(
            preprocessor=texto_para_tfidf,
            ngram_range=(1, 2), min_df=2, max_features=20000,
        )),
        ("numericas", Pipeline([
            ("extrair", FunctionTransformer(transformar_textos, validate=False)),
            ("padronizar", StandardScaler()),
        ])),
    ])


modelos = {
    "logistica": Pipeline([
        ("representacao", criar_representacao()),
        ("classificador", LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=SEMENTE,
        )),
    ]),
    "svm_linear": Pipeline([
        ("representacao", criar_representacao()),
        ("classificador", LinearSVC(
            class_weight="balanced", random_state=SEMENTE, max_iter=10000,
        )),
    ]),
}

referencia = DummyClassifier(strategy="most_frequent")
referencia.fit([[0]] * len(treino), treino["classe"])
prev_ref = referencia.predict([[0]] * len(validacao))
comparacao = [{
    "modelo": "referencia",
    "acuracia": accuracy_score(validacao["classe"], prev_ref),
    "f1_macro": f1_score(validacao["classe"], prev_ref, average="macro", zero_division=0),
}]

for nome, modelo in modelos.items():
    modelo.fit(treino["texto"], treino["classe"])
    previsto = modelo.predict(validacao["texto"])
    comparacao.append({
        "modelo": nome,
        "acuracia": accuracy_score(validacao["classe"], previsto),
        "f1_macro": f1_score(validacao["classe"], previsto, average="macro", zero_division=0),
    })
    print(nome)
    print(classification_report(
        validacao["classe"], previsto, labels=CLASSES, zero_division=0,
    ))

comparacao = pd.DataFrame(comparacao).sort_values("f1_macro", ascending=False)
comparacao.to_csv(RESULTADOS / "comparacao_validacao.csv", index=False)
display(comparacao)
melhor_nome = comparacao[comparacao["modelo"] != "referencia"].iloc[0]["modelo"]
melhor_modelo = modelos[melhor_nome]
print("Candidato escolhido:", melhor_nome)

logistica
                       precision    recall  f1-score   support

 Ensino Fundamental I       0.78      0.91      0.84        44
Ensino Fundamental II       0.65      0.73      0.69        49
         Ensino Médio       0.92      0.73      0.81        92
      Ensino Superior       0.91      0.96      0.94       122

             accuracy                           0.85       307
            macro avg       0.82      0.83      0.82       307
         weighted avg       0.86      0.85      0.85       307

svm_linear
                       precision    recall  f1-score   support

 Ensino Fundamental I       0.82      0.93      0.87        44
Ensino Fundamental II       0.88      0.78      0.83        49
         Ensino Médio       0.98      0.87      0.92        92
      Ensino Superior       0.92      1.00      0.96       122

             accuracy                           0.92       307
            macro avg       0.90      0.89      0.89       307
         weighted avg       0

,modelo,acuracia,f1_macro
2,svm_linear,0.915309,0.894649
1,logistica,0.846906,0.820634
0,referencia,0.397394,0.142191


Candidato escolhido: svm_linear


### Avaliar uma vez no teste e salvar o modelo

In [12]:
previsto_teste = melhor_modelo.predict(teste["texto"])
avaliacao = classification_report(
    teste["classe"], previsto_teste, labels=CLASSES,
    output_dict=True, zero_division=0,
)
(RESULTADOS / "metricas_teste.json").write_text(
    json.dumps(avaliacao, ensure_ascii=False, indent=2), encoding="utf-8",
)
matriz = pd.DataFrame(
    confusion_matrix(teste["classe"], previsto_teste, labels=CLASSES),
    index=CLASSES, columns=CLASSES,
)
matriz.to_csv(RESULTADOS / "matriz_confusao.csv", index_label="real")
display(matriz)
print(classification_report(teste["classe"], previsto_teste, labels=CLASSES, zero_division=0))

(RAIZ / "modelos").mkdir(exist_ok=True)
joblib.dump({
    "pipeline": melhor_modelo,
    "nome": melhor_nome,
    "classes": CLASSES,
    "semente": SEMENTE,
    "sklearn_version": sklearn.__version__,
    "unidade_treino": "documento_inteiro",
}, RAIZ / "modelos" / "modelo.joblib")

carregado = joblib.load(RAIZ / "modelos" / "modelo.joblib")
exemplo = "A menina estava feliz. Ela foi brincar na escola."
assert carregado["pipeline"].predict([exemplo])[0] == melhor_modelo.predict([exemplo])[0]
print("Previsão de exemplo:", carregado["pipeline"].predict([exemplo])[0])

,Ensino Fundamental I,Ensino Fundamental II,Ensino Médio,Ensino Superior
Ensino Fundamental I,41,2,1,0
Ensino Fundamental II,10,34,4,0
Ensino Médio,0,0,90,3
Ensino Superior,1,0,3,119


                       precision    recall  f1-score   support

 Ensino Fundamental I       0.79      0.93      0.85        44
Ensino Fundamental II       0.94      0.71      0.81        48
         Ensino Médio       0.92      0.97      0.94        93
      Ensino Superior       0.98      0.97      0.97       123

             accuracy                           0.92       308
            macro avg       0.91      0.89      0.89       308
         weighted avg       0.93      0.92      0.92       308

Previsão de exemplo: Ensino Fundamental I
